# `semantic.v_leaderboard` — view

Thin view over Gold. No logic beyond shaping.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [0]:
-- THE LEADERBOARD. Every ticker at every horizon, with SPY ranked among them so its
-- position is always visible. Display only: the dashboard sorts and limits this view, and
-- the beat rate above never reads it. Ranking the fact and then filtering it would make
-- "what % beat the index" circular -- the answer would always be 100%.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_leaderboard
COMMENT 'Every ticker at every horizon, ranked by return, income and risk-adjusted return'
AS
SELECT f.horizon_years,
       f.ticker,
       CASE WHEN d.entity_type = 'Index' THEN CONCAT(d.trust_name, ' (the index)')
            ELSE d.trust_name END                        AS name,
       d.entity_type,
       d.management_group,
       d.manager,
       d.manager_structure,
       d.aic_sector,
       f.rank_by_return,
       f.rank_by_income,
       f.rank_by_risk_adjusted,
       ROUND(100 * f.total_return, 1)                    AS total_return_pct,
       ROUND(100 * f.annualised_return, 2)               AS annualised_return_pct,
       ROUND(100 * f.income_return, 1)                   AS income_pct,
       ROUND(100 * f.volatility, 1)                      AS volatility_pct,
       ROUND(f.risk_adjusted_return, 2)                  AS risk_adjusted_return,
       ROUND(100 * f.index_return_same_period, 1)        AS index_return_pct,
       f.beat_index,
       f.months_used,
       f.return_basis,
       d.source_url
FROM `index-vs-trust-pipeline`.gold.fact_horizon_performance f
JOIN `index-vs-trust-pipeline`.gold.dim_ticker d ON d.ticker_key = f.ticker_key;

## Verification

Expected: **440 rows** — every ticker at every horizon.

In [0]:
SELECT COUNT(*) AS rows
FROM `index-vs-trust-pipeline`.semantic.v_leaderboard;